In [1]:
from pyscf import gto, scf, cc

a = 2 # bond length in a cluster
d = 100 # distance between each cluster
unit = 'b' # unit of length
na = 2 # size of a cluster (monomer)
nc = 4 # set as integer multiple of monomers
spin = 0 # spin per monomer
frozen = 0 # frozen orbital per monomer
elmt = 'H'
basis = 'sto6g'
# for nc in nc_list:
atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms, basis="sto6g", unit='B', spin=0, verbose=4)
mol.build()

mf = scf.RHF(mol)
mf.kernel()

mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)
mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)

nfrozen = 0
mycc = cc.CCSD(mf,frozen=nfrozen)
mycc.kernel()[0]

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-19-generic', version='#19~24.04.2-Ubuntu SMP PREEMPT_DYNAMIC Fri Mar  6 23:08:46 UTC 2', machine='x86_64')  Threads 16
Python 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]
numpy 2.3.1  scipy 1.16.2  h5py 3.14.0
Date: Fri Mar 27 16:04:55 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 8
[INPUT] num. electrons = 8
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 H      0.000000000000   0.000000000000   0.000000000000 AA    

np.float64(-0.15856560367764583)

In [8]:
# example for PT2

options = {'n_eql': 3,
           'n_prop_steps': 50,
            'n_ene_blocks': 1,
            'n_sr_blocks': 5,
            'n_blocks': 50,
            'n_walkers': 1,
            'seed': 17,
            'walker_type': 'rhf',
            'trial': 'stoccsd',
            'nslater': 10,
            'dt':0.005,
            'use_gpu': False,
            }

from ad_afqmc.prop_unrestricted.mixed_wave import prep
import jax
jax.config.update("jax_enable_x64", True)
prep.prep_afqmc(mycc,chol_cut=1e-5)
# prop_unrestricted.run_afqmc(options,nproc=1)
option_file='options.bin'
import pickle
with open(option_file, 'wb') as f:
    pickle.dump(options, f)

#
# Preparing AFQMC calculation
# Calculating Cholesky integrals
# Finished calculating Cholesky integrals
#
# Size of the correlation space:
# Number of electrons: (4, 4)
# Number of basis functions: 8
# Number of Cholesky vectors: 12
#


In [9]:
import time
import numpy as np
from jax import random
from jax import numpy as jnp
from functools import partial 

ham_data, ham, prop, trial, wave_data, sampler, options = (prep._prep_afqmc())

init_time = time.time()

### initialize propagation
init_walkers = None
trial_rdm1 = trial.get_rdm1(wave_data)
if "rdm1" not in wave_data:
    wave_data["rdm1"] = trial_rdm1
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)

prop_data = prop.init_prop_data(trial, wave_data, ham_data, init_walkers)
if jnp.abs(jnp.sum(prop_data["overlaps"])) < 1.0e-6:
    raise ValueError(
        "Initial overlaps are zero. Pass walkers with non-zero overlap."
    )
prop_data["key"] = random.PRNGKey(options["seed"])

# wave_data['tau'], _ = trial.decompose_t2(wave_data)

prop_data["overlaps"] = trial.calc_overlap(prop_data["walkers"], wave_data)
prop_data["n_killed_walkers"] = 0

e_init= jnp.real(trial.calc_energy(prop_data["walkers"], ham_data, wave_data)[0])
prop_data["e_estimate"] = e_init
prop_data["pop_control_ene_shift"] = prop_data["e_estimate"]

print(e_init)
print(e_init-mf.e_tot)

# Throw 12 vectors in T2 deomposition
# cutoff = 1.00e-08 | error = 2.96e-15
# number of T2 decomposition vectors 4
# nelec: (4, 4)
# norb: 8
# nchol: 12
# n_eql: 3
# n_prop_steps: 50
# n_ene_blocks: 1
# n_sr_blocks: 5
# n_blocks: 50
# n_walkers: 1
# seed: 17
# walker_type: rhf
# trial: stoccsd
# nslater: 10
# dt: 0.005
# use_gpu: False
# n_exp_terms: 6
# n_batch: 1
# max_error: 0.001
-4.225719528251248
3.552713678800501e-15


In [10]:
olp = prop_data["overlaps"]
wt = prop_data["weights"]
xtaus, prop_data = trial.get_xtaus(prop_data, wave_data, prop)
olp_cc, energy_cc = trial.calc_energy_stoccsd(prop_data["walkers"], xtaus, ham_data, wave_data)
estocc = jnp.sum(wt * olp_cc/olp * energy_cc)  / jnp.sum(wt * olp_cc/olp)
print(estocc)
print(mycc.e_tot)

(-4.370374543785198+6.380030125545779e-10j)
-4.384285131928897


In [6]:
from jax import jit, lax

@partial(jit, static_argnums=(0,3,4))
def _block(self, prop_data, ham_data, prop, trial, wave_data):
    """Block scan function. Propagation and energy calculation."""

    prop_data["key"], subkey = random.split(prop_data["key"])
    fields = random.normal(
        subkey,
        shape=(
            self.n_prop_steps,
            prop.n_walkers,
            self.n_chol,
        ),
    )
    _step_scan_wrapper = lambda x, y: self._step_scan(
        x, y, ham_data, prop, trial, wave_data
    )
    prop_data, _ = lax.scan(_step_scan_wrapper, prop_data, fields)
    prop_data["n_killed_walkers"] += prop_data["weights"].size - jnp.count_nonzero(
        prop_data["weights"]
    )

    # raondom fields_x for T2 decomposition
    xtaus, prop_data = trial.get_xtaus(prop_data, wave_data, prop)
    prop_data = prop.orthonormalize_walkers(prop_data)
    # prop_data = prop.stochastic_reconfiguration_local(prop_data)

    olp_hf = trial.calc_overlap(prop_data["walkers"], wave_data)
    prop_data["overlaps"] = olp_hf
    ene_hf = jnp.real(trial.calc_energy(prop_data["walkers"], ham_data, wave_data))
    olp_cc, ene_cc = trial.calc_energy_stoccsd(prop_data["walkers"], xtaus, ham_data, wave_data)
    wt_hf = prop_data["weights"]

    weight = jnp.sum(wt_hf)
    ehf = jnp.sum(wt_hf * ene_hf) / weight
    ecc = jnp.sum(wt_hf * olp_cc / olp_hf * ene_cc) / weight
    occ = jnp.sum(wt_hf * olp_cc / olp_hf) / weight
    occ_abs = jnp.sum(wt_hf * jnp.abs(olp_cc / olp_hf)) / weight

    prop_data = prop.stochastic_reconfiguration_local(prop_data)
    prop_data["overlaps"] = trial.calc_overlap(prop_data["walkers"], wave_data)

    return prop_data, (weight, ehf, ecc, occ, occ_abs)

@partial(jit, static_argnums=(0,3,4))
def _block_froze(self, prop_data, ham_data, prop, trial, wave_data):
    """Block scan function. Propagation and energy calculation."""

    # raondom fields_x for T2 decomposition
    xtaus, prop_data = trial.get_xtaus(prop_data, wave_data, prop)

    olp_hf = trial.calc_overlap(prop_data["walkers"], wave_data)
    prop_data["overlaps"] = olp_hf
    ene_hf = jnp.real(trial.calc_energy(prop_data["walkers"], ham_data, wave_data))
    olp_cc, ene_cc = trial.calc_energy_stoccsd(prop_data["walkers"], xtaus, ham_data, wave_data)
    wt_hf = prop_data["weights"]

    weight = jnp.sum(wt_hf)
    ehf_avg = jnp.sum(wt_hf * ene_hf) / weight
    ecc_avg = jnp.sum(wt_hf * olp_cc / olp_hf * ene_cc) / weight
    occ_avg = jnp.sum(wt_hf * olp_cc / olp_hf) / weight
    occ_abs = jnp.sum(wt_hf * jnp.abs(olp_cc / olp_hf)) / weight

    return prop_data, (weight, ehf_avg, ecc_avg, occ_avg, occ_abs)

In [7]:
trial.nslater = 10
xtaus, prop_data = trial.get_xtaus(prop_data, wave_data, prop)
olp_cc, ene_cc = trial.calc_energy_stoccsd(prop_data["walkers"], xtaus, ham_data, wave_data)
olp_hf = prop_data["overlaps"]
wt_hf = prop_data["weights"]
ecc_init = jnp.real(jnp.sum(wt_hf*olp_cc/olp_hf*ene_cc)/jnp.sum(wt_hf*olp_cc/olp_hf))
print(ecc_init)
print(mycc.e_tot)

-4.384669244164366
-4.384285131928897


In [32]:
from ad_afqmc.prop_unrestricted.mixed_wave import sampling
ehf_init = jnp.real(prop_data["e_estimate"])

den, num = trial.calc_energy_stoccsd(prop_data["walkers"], xtaus, ham_data, wave_data)
ecc_init = jnp.real(jnp.sum(num) / jnp.sum(den))

print(f'# Propagating with {options["n_walkers"]} walkers')
print("# Equilibration sweeps:")
print("# atom_time    ehf    e_stocc    Walltime")
print(f"  {0.:.2f}  {ehf_init:.6f}   {ecc_init:.6f}  {time.time() - init_time:.2f}")

sampler_eq = sampling.sampler_stoccsd(
    n_prop_steps=50, 
    n_ene_blocks=1, 
    n_sr_blocks=1,
    n_chol = sampler.n_chol
    )

block_time = prop.dt * sampler_eq.n_prop_steps * sampler_eq.n_ene_blocks * sampler_eq.n_sr_blocks

for n in range(100):
    prop_data, (whf, ehf, ecc_num, occ_den, occ_abs) \
        = _block(sampler_eq, prop_data, ham_data, prop, trial, wave_data)

    ecc = jnp.real(ecc_num / occ_den)
    sign = occ_den / occ_abs
    sign_real = jnp.real(sign)
    sign_imag = jnp.imag(sign)

    prop_data["e_estimate"] = 0.9 * prop_data["e_estimate"] + 0.1 * ecc

    print(f" {(n+1)*block_time:.2f}  {whf:.6f}  {ehf:.6f}  {ecc_num:.6f}  {occ_den:.6f}  {ecc:.6f}  {sign_real:.6f}  {sign_imag:.6f}  {time.time() - init_time:.2f} ")


# Propagating with 300 walkers
# Equilibration sweeps:
# atom_time    ehf    e_stocc    Walltime
  0.00  -4.225720   -4.382677  16.11
 0.25  300.172979  -4.261140  -4.569852+0.000106j  1.046800-0.000012j  -4.365545  0.999773  -0.000012  18.35 
 0.50  300.499052  -4.290814  -4.742158-0.013016j  1.088604+0.003929j  -4.356171  0.998928  0.003605  20.40 
 0.75  301.029154  -4.320850  -4.745643-0.010266j  1.078203+0.002386j  -4.401436  0.997870  0.002208  20.45 
 1.00  300.879440  -4.330953  -4.713173+0.004475j  1.068325-0.001834j  -4.411734  0.993448  -0.001706  20.51 
 1.25  300.819273  -4.343545  -5.112596+0.014947j  1.176736-0.004108j  -4.344717  0.997034  -0.003481  20.57 
 1.50  301.092622  -4.366197  -5.034165+0.019728j  1.154053-0.006367j  -4.362121  0.996429  -0.005497  20.63 
 1.75  301.058443  -4.381594  -5.120127+0.040808j  1.175945-0.011911j  -4.353957  0.995651  -0.010084  20.69 
 2.00  300.735986  -4.379878  -4.799482+0.023708j  1.082374-0.004559j  -4.434232  0.995997  -0.004

In [ ]:
print("# Sampling sweeps:")
print("# nBlocks  energy/hf  error  energy/cc  error  Walltime")
sampler.n_blocks = 1000

whf_sp = np.zeros(sampler.n_blocks,dtype="float64")
ehf_sp = np.zeros(sampler.n_blocks,dtype="float64")
ecc_num_sp = np.zeros(sampler.n_blocks,dtype="complex128")
occ_den_sp = np.zeros(sampler.n_blocks,dtype="complex128")
occ_abs_sp = np.zeros(sampler.n_blocks,dtype="float64")

for n in range(sampler.n_blocks):
    prop_data, (whf, ehf, ecc_num, occ_den, occ_abs) \
        = _block(sampler_eq, prop_data, ham_data, prop, trial, wave_data)

    whf_sp[n] = whf
    ehf_sp[n] = ehf
    ecc_num_sp[n] = ecc_num
    occ_den_sp[n] = occ_den
    occ_abs_sp[n] = occ_abs
    ecc_estimate = jnp.real(ecc_num / occ_den)
    # sign = occ_den / occ_abs
    # sign_real = jnp.real(sign)
    # sign_imag = jnp.imag(sign)


    prop_data["e_estimate"] = 0.9 * prop_data["e_estimate"] + 0.1 * ecc_estimate

    # if (n+1) % (max(sampler.n_blocks // 10, 1)) == 0 and n > 0:
    if n > 0:
        weight = np.sum(whf_sp[:n+1])
        whf_ehf = np.sum(whf_sp[:n+1] * ehf_sp[:n+1])
        whf_num = np.sum(whf_sp[:n+1] * ecc_num_sp[:n+1])
        whf_den = np.sum(whf_sp[:n+1] * occ_den_sp[:n+1])

        ehf = np.real(whf_ehf / weight)
        ehf_err = np.sqrt(np.sum(whf_sp[:n+1] * (ehf_sp[:n+1]-ehf)**2 / weight)) / np.sqrt(n)

        ecc = np.real(whf_num / whf_den)
        ecc_sp = (whf_sp[:n+1] * ecc_num_sp[:n+1]) / (whf_sp[:n+1] * occ_den_sp[:n+1])
        ecc_err = np.real(np.std(ecc_sp) / np.sqrt(len(ecc_sp)))

        occ_den = np.sum(whf_sp[:n+1] * occ_den_sp[:n+1]) / weight
        occ_abs = np.sum(whf_sp[:n+1] * occ_abs_sp[:n+1]) / weight
        sign = occ_den / occ_abs
        sign_real = jnp.real(sign)
        sign_imag = jnp.imag(sign)

        print(f" {n+1:4d}  {ehf:.6f}  {ehf_err:.6f}  {ecc:.6f}  {ecc_err:.6f}  {sign_real:.6f}  {sign_imag:.6f}  {time.time() - init_time:.2f}")

# Sampling sweeps:
# nBlocks  energy/hf  error  energy/cc  error  Walltime
    2  -4.371764  0.007588  -4.422533  0.018137  0.995818  -0.009366  45.97
    3  -4.368334  0.005563  -4.411415  0.015661  0.995763  -0.005009  46.03
    4  -4.370727  0.004604  -4.407419  0.012330  0.995667  -0.004459  46.09
    5  -4.372932  0.004192  -4.407308  0.010014  0.995281  -0.004415  46.15
    6  -4.373551  0.003478  -4.405121  0.008585  0.995272  -0.003798  46.21
    7  -4.373653  0.002942  -4.408060  0.007836  0.995458  -0.003550  46.26
    8  -4.372324  0.002873  -4.402398  0.008670  0.995546  -0.003352  46.32
    9  -4.371759  0.002596  -4.404090  0.007910  0.995629  -0.002358  46.38
   10  -4.370609  0.002591  -4.402737  0.007348  0.995693  -0.001209  46.44
   11  -4.370472  0.002348  -4.398647  0.007750  0.995694  -0.002435  46.50
   12  -4.370974  0.002201  -4.395672  0.007628  0.995754  -0.002559  46.56
   13  -4.373664  0.003367  -4.386996  0.010307  0.995799  -0.002471  46.62
   14  -4.373

In [34]:
mycc.e_tot

np.float64(-4.384285131928896)

In [35]:
ecc_err = sampler.blk_average(whf_sp, ecc_num_sp, occ_den_sp, max_size=None)

# Blk_SZ  NBlk  NSmp  Energy  Error
 1  2000  2000  -4.387192  0.001291
 2  1000  2000  -4.385697  0.001465
 3  666  1998  -4.384832  0.001549
 4  500  2000  -4.384575  0.001645
 5  400  2000  -4.384316  0.001699
 6  333  1998  -4.384149  0.001793
 7  285  1995  -4.383965  0.001808
 8  250  2000  -4.383918  0.001873
 9  222  1998  -4.383831  0.001921
 10  200  2000  -4.383738  0.001926
 11  181  1991  -4.383711  0.001984
 12  166  1992  -4.383584  0.001987
 13  153  1989  -4.383572  0.001966
 14  142  1988  -4.383514  0.001984
 15  133  1995  -4.383484  0.002056
 16  125  2000  -4.383480  0.002091
 17  117  1989  -4.383446  0.002049
 18  111  1998  -4.383389  0.002052
 19  105  1995  -4.383335  0.002034
 20  100  2000  -4.383327  0.002013
 21  95  1995  -4.383334  0.002155
 22  90  1980  -4.383338  0.002092
 23  86  1978  -4.383369  0.002129
 24  83  1992  -4.383238  0.002075
 25  80  2000  -4.383264  0.002074
 26  76  1976  -4.383249  0.002119
 27  74  1998  -4.383222  0.002091
 28  7

In [12]:
print("# Frozen Walker Sampling sweeps:")
print("# nBlocks  energy/hf  error  energy/cc  error  Walltime")
sampler.n_blocks = 2000

whf_sp = np.zeros(sampler.n_blocks,dtype="float64")
ehf_sp = np.zeros(sampler.n_blocks,dtype="float64")
ecc_num_sp = np.zeros(sampler.n_blocks,dtype="complex128")
occ_den_sp = np.zeros(sampler.n_blocks,dtype="complex128")
occ_abs_sp = np.zeros(sampler.n_blocks,dtype="float64")

for n in range(sampler.n_blocks):
    prop_data, (whf, ehf, ecc_num, occ_den, occ_abs) \
        = _block_froze(sampler, prop_data, ham_data, prop, trial, wave_data)

    whf_sp[n] = whf
    ehf_sp[n] = ehf
    ecc_num_sp[n] = ecc_num
    occ_den_sp[n] = occ_den
    occ_abs_sp[n] = occ_abs
    ecc_estimate = jnp.real(ecc_num / occ_den)

    prop_data["e_estimate"] = 0.9 * prop_data["e_estimate"] + 0.1 * ecc_estimate

    # if (n+1) % (max(sampler.n_blocks // 10, 1)) == 0 and n > 0:
    if n > 0:
        weight = np.sum(whf_sp[:n+1])
        whf_ehf = np.sum(whf_sp[:n+1] * ehf_sp[:n+1])
        whf_num = np.sum(whf_sp[:n+1] * ecc_num_sp[:n+1])
        whf_den = np.sum(whf_sp[:n+1] * occ_den_sp[:n+1])

        ehf = np.real(whf_ehf / weight)
        ehf_err = np.sqrt(np.sum(whf_sp[:n+1] * (ehf_sp[:n+1]-ehf)**2 / weight)) / np.sqrt(n)

        ecc = np.real(whf_num / whf_den)
        ecc_sp = (whf_sp[:n+1] * ecc_num_sp[:n+1]) / (whf_sp[:n+1] * occ_den_sp[:n+1])
        ecc_err = np.real(np.std(ecc_sp) / np.sqrt(len(ecc_sp)))

        occ_den = np.sum(whf_sp[:n+1] * occ_den_sp[:n+1]) / weight
        occ_abs = np.sum(whf_sp[:n+1] * occ_abs_sp[:n+1]) / weight
        sign = occ_den / occ_abs
        sign_real = jnp.real(sign)
        sign_imag = jnp.imag(sign)

        print(f" {n+1:4d}  {ehf:.6f}  {ehf_err:.6f}  {ecc:.6f}  {ecc_err:.6f}  {sign_real:.6f}  {sign_imag:.6f}  {time.time() - init_time:.2f}")

# Frozen Walker Sampling sweeps:
# nBlocks  energy/hf  error  energy/cc  error  Walltime
    2  -4.225720  0.000000  -4.391703  0.004074  1.000000  0.000000  40.43
    3  -4.225720  0.000000  -4.410186  0.015334  1.000000  0.000000  40.44
    4  -4.225720  0.000000  -4.394119  0.018052  1.000000  0.000000  40.44
    5  -4.225720  0.000000  -4.394640  0.014449  1.000000  0.000000  40.45
    6  -4.225720  0.000000  -4.395740  0.012082  1.000000  0.000000  40.45
    7  -4.225720  0.000000  -4.399408  0.010899  1.000000  0.000000  40.46
    8  -4.225720  0.000000  -4.398876  0.009549  1.000000  0.000000  40.46
    9  -4.225720  0.000000  -4.399439  0.008505  1.000000  0.000000  40.47
   10  -4.225720  0.000000  -4.395651  0.008456  1.000000  0.000000  40.47
   11  -4.225720  0.000000  -4.395054  0.007709  1.000000  0.000000  40.48
   12  -4.225720  0.000000  -4.393326  0.007257  1.000000  0.000000  40.48
   13  -4.225720  0.000000  -4.395896  0.007140  1.000000  0.000000  40.49
   14  -4.2

In [13]:
ecc_err = sampler.blk_average(whf_sp, ecc_num_sp, occ_den_sp, max_size=None)

# Blk_SZ  NBlk  NSmp  Energy  Error
 1  2000  2000  -4.385469  0.000798
 2  1000  2000  -4.385469  0.000786
 3  666  1998  -4.385434  0.000782
 4  500  2000  -4.385469  0.000770
 5  400  2000  -4.385469  0.000750
 6  333  1998  -4.385434  0.000785
 7  285  1995  -4.385413  0.000788
 8  250  2000  -4.385469  0.000751
 9  222  1998  -4.385434  0.000747
 10  200  2000  -4.385469  0.000755
 11  181  1991  -4.385415  0.000699
 12  166  1992  -4.385418  0.000750
 13  153  1989  -4.385423  0.000728
 14  142  1988  -4.385415  0.000743
 15  133  1995  -4.385413  0.000712
 16  125  2000  -4.385469  0.000732
 17  117  1989  -4.385423  0.000747
 18  111  1998  -4.385434  0.000736
 19  105  1995  -4.385413  0.000710
 20  100  2000  -4.385469  0.000742
 21  95  1995  -4.385413  0.000731
 22  90  1980  -4.385387  0.000682
 23  86  1978  -4.385393  0.000782
 24  83  1992  -4.385418  0.000749
 25  80  2000  -4.385469  0.000752
 26  76  1976  -4.385374  0.000710
 27  74  1998  -4.385434  0.000672
 28  7

In [14]:
trial.nslater = 10
xtaus, prop_data = trial.get_xtaus(prop_data, wave_data, prop)
olp_cc, ene_cc = trial.calc_energy_stoccsd(prop_data["walkers"], xtaus, ham_data, wave_data)
olp_hf = prop_data["overlaps"]
wt_hf = prop_data["weights"]
ecc_init = jnp.real(jnp.sum(wt_hf*olp_cc/olp_hf*ene_cc)/jnp.sum(wt_hf*olp_cc/olp_hf))
print(ecc_init)
print(mycc.e_tot)

-4.360260598316588
-4.384285131928897


In [16]:
olp_cc

Array([10.+0.j], dtype=complex128)